In [0]:
import re
from pyspark.sql.functions import current_timestamp

# Define the source directory
BASE = "/Volumes/project/bronze/raw_data"

def ingest_tab_csv(file_name: str, table_name: str):
    path = f"{BASE}/{file_name}"
    
    # 1. Read CSV with tab separator
    df = (spark.read
          .option("header", True)
          .option("sep", "\t")
          .option("inferSchema", True)
          .csv(path))

    # 2. Normalize columns (CamelCase to snake_case)
    new_columns = []
    for c in df.columns:
        # Insert underscore before uppercase letters (e.g., SalesKey -> Sales_Key)
        clean = re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', c.strip())
        # Replace spaces/special chars with underscore and convert to lowercase
        clean = re.sub(r'[\s\t\n\-]+', '_', clean).lower()
        # Remove any double underscores
        clean = re.sub(r'_+', '_', clean)
        new_columns.append(clean)
        
    df = df.toDF(*new_columns)

    # 3. Add ingestion metadata
    df = df.withColumn("ingestion_timestamp", current_timestamp())

    # 4. Save to Bronze layer
    (df.write
      .mode("overwrite")
      .option("overwriteSchema", "true")
      .format("delta")
      .saveAsTable(f"project.bronze.{table_name}"))

    print(f"Success: {table_name} | Rows: {df.count()} | Cols: {len(df.columns)}")


# List of ingestion jobs
jobs = [
    ("Region.csv",            "region"),
    ("SalespersonRegion.csv", "salesperson_region"),
    ("Salesperson.csv",       "salesperson"),
    ("Reseller.csv",          "reseller"),
    ("Product.csv",           "product"),
    ("Targets.csv",           "targets"),
    ("Sales_2017.csv",        "sales_2017"),
    ("Sales_2018.csv",        "sales_2018"),
    ("Sales_2019.csv",        "sales_2019"),
    ("Sales_2020.csv",        "sales_2020"),
]

# Run ingestion loop
for file_name, table_name in jobs:
    ingest_tab_csv(file_name, table_name)